In [1]:
import requests
import csv
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
url = "https://books.toscrape.com/"

# Identify the crawler
headers = {
    "User-Agent": "CollegePracticalCrawler/1.0"
}

products = []
visited_pages = 0
max_pages = 3

try:
    while url and visited_pages < max_pages:
        print("Crawling:", url)

        response = requests.get(
            url, headers=headers, timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        response.encoding = response.apparent_encoding

        # Find all product cards on the page
        product_cards = soup.select("article.product_pod")

        for card in product_cards:
            link_tag = card.select_one("h3 a")
            price_tag = card.select_one(".price_color")
            stock_tag = card.select_one(".availability")

            if link_tag is None:
                continue

            product_name = link_tag.get("title", link_tag.get_text(strip=True))
            product_link = urljoin(url, link_tag.get("href", ""))

            price = (
                price_tag.get_text(strip=True)
                if price_tag else "Not available"
            )

            availability = (
                stock_tag.get_text(" ", strip=True)
                if stock_tag else "Unknown"
            )

            products.append({
                "Product Name": product_name,
                "Price": price,
                "Availability": availability,
                "Product Link": product_link
            })

        visited_pages += 1
        print("Products found:", len(product_cards))

        # Find the next page
        next_link = soup.select_one("li.next a")

        if next_link:
            url = urljoin(url, next_link.get("href", ""))
            time.sleep(1)
        else:
            url = None

    # Save extracted data to CSV
    with open(
        "products.csv", "w", newline="", encoding="utf-8"
    ) as file:
        fieldnames = [
            "Product Name", "Price",
            "Availability", "Product Link"
        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(products)

    print("\nCrawling completed!")
    print("Pages visited:", visited_pages)
    print("Total products collected:", len(products))
    print("Data saved to products.csv")

except requests.RequestException as error:
    print("Website request failed:", error)


Crawling: https://books.toscrape.com/
Products found: 20
Crawling: https://books.toscrape.com/catalogue/page-2.html
Products found: 20
Crawling: https://books.toscrape.com/catalogue/page-3.html
Products found: 20

Crawling completed!
Pages visited: 3
Total products collected: 60
Data saved to products.csv
